####  Bronze vs Silver Testing – payment_methods Table

This document describes the data quality, integrity, and reconciliation tests performed for the `payment_methods` table during ingestion from Bronze to Silver.





####  Tables Under Test

- Bronze: `coffee.bronze.payment_methods`
- Silver: `coffee.silver.payment_methods`


In [0]:
-- Test 1: Compare row counts between Bronze and Silver
-- Silver count should be <= Bronze count due to deduplication and filtering

SELECT 'bronze' AS layer, COUNT(*) AS record_count
FROM coffee.bronze.payment_methods

UNION ALL

SELECT 'silver' AS layer, COUNT(*) AS record_count
FROM coffee.silver.payment_methods;

In [0]:
-- Test 2: Ensure mandatory columns are NOT NULL in Silver
-- method_id, method_name, and category must be present

SELECT COUNT(*) AS invalid_silver_records
FROM coffee.silver.payment_methods
WHERE
  method_id IS NULL
  OR method_name IS NULL
  OR category IS NULL;


In [0]:
-- Test 3: Ensure each payment_method_id appears only once in Silver
-- Confirms deduplication logic is working correctly

SELECT method_id, COUNT(*) AS cnt
FROM coffee.silver.payment_methods
GROUP BY method_id
HAVING COUNT(*) > 1;


In [0]:
-- Test 5: All valid Bronze payment methods should be present in Silver
-- Identifies valid records accidentally dropped

SELECT method_id
FROM coffee.bronze.payment_methods
WHERE
  method_id IS NOT NULL
  AND method_name IS NOT NULL
  AND category IS NOT NULL

EXCEPT

SELECT method_id
FROM coffee.silver.payment_methods;
